# 12 · Train / Validation / Test Split

**Purpose**: Time-based splitting + leakage-safe target re-encoding

| I/O | Path |
|-----|------|
| Input  | `data/processed/features.parquet` |
| Input  | `data/processed/feature_manifest.json` |
| Output | `data/processed/train.parquet` |
| Output | `data/processed/val.parquet` |
| Output | `data/processed/test.parquet` |
| Output | `data/processed/feature_manifest.json` (updated) |

## Split design
- **Test** : last 14 days of data
- **Val**  : 14 days immediately before test
- **Train**: everything before val

## Leakage fix
Notebook 11 computed `territory_median_atd` and `geo_archetype_median_atd`
on the full dataset.  This notebook recomputes them from **train rows only**
and maps those values to val and test, filling unseen categories with the
train global median.

---
## 1 · Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

FEAT_PATH     = Path('../data/processed/features.parquet')
MANIFEST_PATH = Path('../data/processed/feature_manifest.json')
TRAIN_PATH    = Path('../data/processed/train.parquet')
VAL_PATH      = Path('../data/processed/val.parquet')
TEST_PATH     = Path('../data/processed/test.parquet')

HOLDOUT_DAYS = 14  # test window
VAL_DAYS     = 14  # validation window

---
## 2 · Load & Sort

In [ ]:
with open(MANIFEST_PATH, 'r', encoding='utf-8') as fh:
    manifest = json.load(fh)

df = pd.read_parquet(FEAT_PATH)
df['restaurant_offered_timestamp_utc'] = pd.to_datetime(
    df['restaurant_offered_timestamp_utc']
)

# Chronological sort (required for time-based split)
df = df.sort_values(
    'restaurant_offered_timestamp_utc'
).reset_index(drop=True)

ts = df['restaurant_offered_timestamp_utc']
print(f'Rows           : {len(df):,}')
print(f'Columns        : {df.shape[1]}')
print(f'Date range     : {ts.min()} → {ts.max()}')

---
## 3 · Define Split Boundaries

In [ ]:
max_date        = ts.max()
test_start      = max_date - pd.Timedelta(days=HOLDOUT_DAYS)
val_start       = test_start - pd.Timedelta(days=VAL_DAYS)

train_mask = ts < val_start
val_mask   = (ts >= val_start) & (ts < test_start)
test_mask  = ts >= test_start

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print(f'Val  start  : {val_start}')
print(f'Test start  : {test_start}')
print(f'Data end    : {max_date}')
print()
print(
    f'Train : {len(df_train):>9,} rows  '
    f'{df_train["restaurant_offered_timestamp_utc"].min().date()} → '
    f'{df_train["restaurant_offered_timestamp_utc"].max().date()}'
)
print(
    f'Val   : {len(df_val):>9,} rows  '
    f'{df_val["restaurant_offered_timestamp_utc"].min().date()} → '
    f'{df_val["restaurant_offered_timestamp_utc"].max().date()}'
)
print(
    f'Test  : {len(df_test):>9,} rows  '
    f'{df_test["restaurant_offered_timestamp_utc"].min().date()} → '
    f'{df_test["restaurant_offered_timestamp_utc"].max().date()}'
)

---
## 4 · Assert Zero Timestamp Overlap

In [ ]:
train_ts = set(df_train['restaurant_offered_timestamp_utc'].astype(str))
val_ts   = set(df_val['restaurant_offered_timestamp_utc'].astype(str))
test_ts  = set(df_test['restaurant_offered_timestamp_utc'].astype(str))

overlap_tv = train_ts & val_ts
overlap_vt = val_ts & test_ts
overlap_tt = train_ts & test_ts

assert len(overlap_tv) == 0, (
    f'Train/Val overlap: {len(overlap_tv)} timestamps'
)
assert len(overlap_vt) == 0, (
    f'Val/Test overlap: {len(overlap_vt)} timestamps'
)
assert len(overlap_tt) == 0, (
    f'Train/Test overlap: {len(overlap_tt)} timestamps'
)

print('No timestamp overlap between splits.')
print(
    f'Coverage check: {len(df_train) + len(df_val) + len(df_test):,}'
    f' == {len(df):,} → {len(df_train)+len(df_val)+len(df_test)==len(df)}'
)

---
## 5 · Fix Target-Encoding Leakage

Notebook 11 computed `territory_median_atd` and `geo_archetype_median_atd`
on the full dataset.  We now recompute them from **train only** and apply
those mappings to val and test.  Unseen categories receive the train global
median.

In [ ]:
train_global_median = df_train['ATD'].median()

# ── territory_median_atd ──────────────────────────────────────────────────
territory_map = (
    df_train.groupby('territory', observed=True)['ATD']
    .median()
)

for split in [df_train, df_val, df_test]:
    split['territory_median_atd'] = (
        split['territory']
        .map(territory_map)
        .astype(float)
        .fillna(train_global_median)
    )

# ── geo_archetype_median_atd ──────────────────────────────────────────────
geo_map = (
    df_train.groupby('geo_archetype', observed=True)['ATD']
    .median()
)

for split in [df_train, df_val, df_test]:
    split['geo_archetype_median_atd'] = (
        split['geo_archetype']
        .map(geo_map)
        .astype(float)
        .fillna(train_global_median)
    )

# ── territory_flow_median_atd ─────────────────────────────────────────────
if 'territory_flow_median_atd' in df_train.columns:
    flow_map = (
        df_train.groupby(
            ['territory', 'courier_flow'], observed=True
        )['ATD'].median()
    )
    for split in [df_train, df_val, df_test]:
        split['territory_flow_median_atd'] = (
            split.set_index(['territory', 'courier_flow'])
            .index.map(flow_map)
            .astype(float)
        )
        split['territory_flow_median_atd'] = (
            split['territory_flow_median_atd'].fillna(train_global_median)
        )

# ── territory_hour_median_atd ─────────────────────────────────────────────
if 'territory_hour_median_atd' in df_train.columns:
    hour_map = (
        df_train.groupby(
            ['territory', 'hour_local'], observed=True
        )['ATD'].median()
    )
    for split in [df_train, df_val, df_test]:
        split['territory_hour_median_atd'] = (
            split.set_index(['territory', 'hour_local'])
            .index.map(hour_map)
            .astype(float)
        )
        split['territory_hour_median_atd'] = (
            split['territory_hour_median_atd'].fillna(train_global_median)
        )

print('Leakage fix applied.')
print(f'Train global ATD median : {train_global_median:.2f} min')
print(f'Territory values mapped : {len(territory_map)}')
print(f'Geo-archetype values    : {len(geo_map)}')

# Sanity: val/test unseen categories
for split_name, split in [('val', df_val), ('test', df_test)]:
    unseen_terr = (
        ~split['territory']
        .isin(territory_map.index)
    ).sum()
    unseen_geo = (
        ~split['geo_archetype']
        .isin(geo_map.index)
    ).sum()
    print(
        f'{split_name}: {unseen_terr} unseen territory rows, '
        f'{unseen_geo} unseen geo_archetype rows (→ global median)'
    )

---
## 6 · ATD Distribution Across Splits

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (name, split) in zip(
    axes,
    [('Train', df_train), ('Val', df_val), ('Test', df_test)]
):
    ax.hist(split['ATD'].clip(upper=120), bins=60,
            color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(split['ATD'].median(), color='red',
               linestyle='--', linewidth=1.5,
               label=f'Median {split["ATD"].median():.1f}')
    ax.set_title(
        f'{name}  ({len(split):,} rows)'
    )
    ax.set_xlabel('ATD (min, clipped at 120)')
    ax.legend(fontsize=9)

plt.suptitle('ATD Distribution by Split', fontweight='bold')
plt.tight_layout()
plt.show()

print('ATD summary per split:')
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    s = split['ATD']
    print(
        f'  {name:<6} mean={s.mean():.1f}  '
        f'median={s.median():.1f}  '
        f'p90={s.quantile(0.9):.1f}  '
        f'sla_rate={( s <= 45).mean()*100:.1f}%'
    )

---
## 7 · Update Manifest & Save Parquet Files

In [ ]:
# Update manifest with split dates
manifest['split_date_val_start']  = str(val_start.date())
manifest['split_date_test_start'] = str(test_start.date())
manifest['split_date_end']        = str(max_date.date())

with open(MANIFEST_PATH, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2)
print('feature_manifest.json updated.')

# Save splits
df_train.to_parquet(TRAIN_PATH, index=False, engine='pyarrow')
df_val.to_parquet(VAL_PATH,   index=False, engine='pyarrow')
df_test.to_parquet(TEST_PATH,  index=False, engine='pyarrow')

for name, path, split in [
    ('Train', TRAIN_PATH, df_train),
    ('Val',   VAL_PATH,   df_val),
    ('Test',  TEST_PATH,  df_test),
]:
    size_mb = path.stat().st_size / 1024 ** 2
    print(
        f'{name:<6} → {path.name:<20}  '
        f'{split.shape}  {size_mb:.1f} MB'
    )